In [3]:
%pip install pytest

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pytest
from pyspark.sql import SparkSession


def _create_spark_session():
    return SparkSession.builder \
    .appName('pytest') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

    spark.sparkContext.setLogLevel("ERROR")


@pytest.fixture(scope="session")
def spark():
    return _create_spark_session() 


def test_column_unique(spark, target_table):
    """ Test column unique values"""
    
    src= spark.sql(f"""
        SELECT 
            COUNT(*) 
        FROM 
            (SELECT product_id, COUNT(*) c 
        FROM 
            {target_table} 
        GROUP BY 
            product_id HAVING c > 1)
    """).take(1)[0][0]
    
    expected = 0
    
    assert src == expected, f"❌ Existem {src} IDs duplicados"


def test_column_not_null(spark, target_table):
    """ Test if column not null"""
    
    src = spark.sql(f"""
        SELECT 
            COUNT(*) as n 
        FROM 
            {target_table} 
        WHERE 
            product_name IS NULL
    """).take(1)[0]["n"]
    
    expected = 0
    
    assert src == expected, f"❌ Existem {src} product_name nulos"
    

def test_schema_validations(spark, target_table):
    """ Test table's schema"""
    
    src = spark.table(target_table).dtypes

    expected = [
        ('product_id', 'string'),
        ('product_name', 'string'),
        ('category', 'string'),
        ('price', 'double')
    ]

    assert src == expected, f"❌ Schema {src} é diferente do esperado"


### Com armazenamento de logs dos testes

In [5]:
import traceback
import inspect
import time
from datetime import datetime

def run_tests(test_funcs, fixture_funcs=None, target_table=None, log_path=None):
    """
    Runner estilo pytest com:
    - Injeção de fixtures
    - Saída formatada
    - Registro de logs no DataFrame Spark e opcionalmente salva como Iceberg ou delta
    """
    fixtures = {}
    logs = []

    
    if fixture_funcs:
        if isinstance(fixture_funcs, dict):
            for name, fx in fixture_funcs.items():
                fixtures[name] = fx()
        elif isinstance(fixture_funcs, list):
            for fx in fixture_funcs:
                fixtures[fx.__name__] = fx()

    
    if target_table:
        fixtures['target_table'] = target_table

    for func in test_funcs:
        start = time.time()
        test_name = func.__name__
        desc = (func.__doc__ or "").strip()
        try:
            sig = inspect.signature(func)
            kwargs = {
                pname: fixtures[pname]
                for pname in sig.parameters
                if pname in fixtures
            }

            func(**kwargs)

            duration = time.time() - start
            print(f"✅ {test_name} PASSED ({duration:.2f}s)")
            # if desc:
            #    print(f"📘 {desc}")
            logs.append((test_name, "PASSED", "", duration, datetime.now()))

        except AssertionError as e:
            duration = time.time() - start
            print(f"❌ {test_name} FAILED: {e} ({duration:.2f}s)")
            logs.append((test_name, "FAILED", str(e), duration, datetime.now()))

        except Exception as e:
            duration = time.time() - start
            print(f"💥 {test_name} ERROR: ({duration:.2f}s)")
            traceback.print_exc()
            logs.append((test_name, "ERROR", str(e), duration, datetime.now()))

    
    spark = fixtures.get("spark")
    if spark and logs:
        log_df = spark.createDataFrame(logs, ["test_name", "status", "message", "duration", "timestamp"]) 
        

        if log_path:
            log_df.write.mode("append").format("iceberg").save(log_path)
            print(f"\n📁 Logs salvos em: {log_path}")
        else:
            print("\n📋 Resultado dos testes:")
            log_df.show()


In [6]:
target_table='iceberg.silver.tbl_silver_product_catalog'


run_tests(
    test_funcs=[test_column_unique, test_column_not_null, test_schema_validations],
    fixture_funcs={'spark': _create_spark_session},
    target_table=target_table,
    log_path=None  # None to show in notbooks, path to write in iceberg or delta format
)


25/11/16 15:55:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/16 15:56:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/16 15:56:01 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/11/16 15:56:01 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/11/16 15:56:01 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/11/16 15:56:01 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/11/16 15:56:01 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J:

✅ test_column_unique PASSED (13.53s)
✅ test_column_not_null PASSED (0.47s)
✅ test_schema_validations PASSED (0.04s)

📋 Resultado dos testes:


+--------------------+------+-------+--------------------+--------------------+
|           test_name|status|message|            duration|           timestamp|
+--------------------+------+-------+--------------------+--------------------+
|  test_column_unique|PASSED|       |  13.531828880310059|2025-11-16 15:56:...|
|test_column_not_null|PASSED|       | 0.47371387481689453|2025-11-16 15:56:...|
|test_schema_valid...|PASSED|       |0.042545318603515625|2025-11-16 15:56:...|
+--------------------+------+-------+--------------------+--------------------+

